# Phase 3: Content Loss Validation
In this notebook, we mathematically test our Content Loss function to see if it correctly penalizes images that lose the structural integrity of Conor McGregor.

In [ ]:
import sys
import os
import torch

# Add src to path
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from data.image_loader import load_image
from models.content_encoder import ContentEncoder
from training.losses import calc_content_loss

In [ ]:
# 1. Setup the GPU and the Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
content_encoder = ContentEncoder().to(device)

# 2. Load the Original Image (McGregor)
mcgregor = load_image('../data/content_images/mcgregor.jpg', max_size=400).to(device)

# 3. Load the Style Image (Icarus - to test something completely different)
icarus = load_image('../data/style_images/icarus.jpg', max_size=400).to(device)

# 4. Create a "Noisy" McGregor (Simulating a badly generated image)
# We add random mathematical noise to the original tensor
noisy_mcgregor = mcgregor + (torch.randn_like(mcgregor) * 0.5)
noisy_mcgregor = torch.clamp(noisy_mcgregor, 0, 1) # Keep pixels between 0 and 1

In [ ]:
# 5. Extract the Deep Content Features (conv4_2)
features_original = content_encoder(mcgregor)
features_icarus = content_encoder(icarus)
features_noisy = content_encoder(noisy_mcgregor)

print(f"Feature map shape: {features_original.shape}")

### The Math Test
Let's calculate the Mean Squared Error (MSE) between our original image and the others.

In [ ]:
# Test 1: McGregor vs McGregor
# This should be exactly 0, because there is 0 difference between the two images.
loss_identical = calc_content_loss(features_original, features_original)
print(f"Loss (McGregor vs McGregor): {loss_identical.item():.4f}")

# Test 2: McGregor vs Noisy McGregor
# The image still vaguely looks like him, but it's corrupted. 
# The network should output a medium-sized penalty.
loss_noisy = calc_content_loss(features_noisy, features_original)
print(f"Loss (McGregor vs Noisy McGregor): {loss_noisy.item():.4f}")

# Test 3: McGregor vs Icarus Painting
# The painting has absolutely none of McGregor's structural features (no human face). 
# The network should output a MASSIVE penalty.
loss_different = calc_content_loss(features_icarus, features_original)
print(f"Loss (McGregor vs Icarus): {loss_different.item():.4f}")